# Square-Root Law ABM — reproduction walkthrough

Reproduces the key results of **arXiv:2607.04280**, "Order Splitting and Liquidity Replenishment Are Jointly Necessary for the Square-Root Law of Market Impact."

This notebook runs a *small* version of the experiments (a handful of stocks, shorter trajectories) so it completes in a few minutes — not the paper's full 2000-stock / 1e6-step configuration. See `README.md` for how to run the full-scale reproduction via `train.py`.

**Reproducibility note:** several of the paper's implementation details (HFT adaptive-spread rule, news-agent trigger process, exact ablation-scenario parameters) are not specified in the paper text and were filled in with literature-standard defaults, flagged `# ASSUMED` in `configs/config.yaml`. Do not expect this notebook's numbers to exactly match Table 1 out of the box — see the README's *Reproducibility Notes* section.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from sqrt_law_abm.utils.config import Config, set_global_seed
from sqrt_law_abm.data.dataset import StockParameterSampler, apply_scenario
from sqrt_law_abm.data.transforms import MetaorderReconstructor
from sqrt_law_abm.models.market import StockMarketSimulation
from sqrt_law_abm.evaluation.metrics import ImpactCurveFitter, TailExponentEstimator, TheoryPredictors

cfg = Config.load('../configs/config.yaml')
set_global_seed(cfg.get('seed', default=42))
print('Config loaded. Sections:', list(cfg.raw.keys()))

## 1. Run one baseline stock (small scale)

The paper uses T=1e6 steps + 5e4 warmup per stock. Here we use a much shorter trajectory so this notebook finishes quickly; expect noisier fits than the paper's Figure 3.

In [ ]:
N_STEPS_DEMO = 60_000   # paper uses 1_000_000
WARMUP_DEMO = 2_000     # paper uses 50_000

sampler = StockParameterSampler(cfg['model'], cfg['simulation'])
stock_config = sampler.sample(n_stocks=1, seed=0)[0]
print(stock_config)

sim = StockMarketSimulation(stock_config.to_dict(), cfg['model'], cfg['simulation'])
result = sim.run(n_steps=N_STEPS_DEMO, warmup_steps=WARMUP_DEMO)
print(f"Recorded {len(result.trade_tape)} trades over {N_STEPS_DEMO} steps")

## 2. Reconstruct metaorders and fit the impact curve (Section 3, Eq. 1-3)

In [ ]:
reconstructor = MetaorderReconstructor()
daily_volume, daily_range = result.daily_volume_and_range()

metaorders = reconstructor.reconstruct(
    result.trade_tape,
    delta_t_ticks=cfg['data']['metaorder_delta_t_ticks'],
    steps_per_day=result.steps_per_day,
)
print(f"Reconstructed {len(metaorders)} metaorders")

q_norm, i_norm = reconstructor.normalize(
    metaorders, daily_volume, daily_range, cfg['data']['min_qnorm_threshold_pct']
)
print(f"{len(q_norm)} valid (Q_norm, I_norm) pairs after filtering")

fitter = ImpactCurveFitter()
fit_result = fitter.log_bin_and_fit(q_norm, i_norm, n_bins=cfg['data']['log_bins'], min_pts=5)
print(f"Fitted: delta={fit_result['delta']:.3f}, c={fit_result['c']:.3f}  (paper baseline: delta=0.549, c=0.982)")

In [ ]:
# Figure 3 style plot: per-stock impact curve
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(q_norm, i_norm, s=4, alpha=0.15, label='individual metaorders')
ax.scatter(fit_result['bin_x'], fit_result['bin_y'], color='C1', label='log-binned means')

x_line = np.logspace(np.log10(q_norm.min()), np.log10(q_norm.max()), 50)
y_line = fit_result['c'] * x_line ** fit_result['delta']
ax.plot(x_line, y_line, color='C1', linestyle='-', label=f"fit: delta={fit_result['delta']:.2f}")

y_half = fit_result['c'] * x_line ** 0.5
ax.plot(x_line, y_half, color='gray', linestyle='--', label='delta=1/2 reference')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Q / V_D'); ax.set_ylabel('I / sigma_D')
ax.set_title('Impact curve (demo scale, cf. paper Figure 3)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Compare against the three theory predictions (Section 4.4, Eqs. 4-5)

In [ ]:
tail_estimator = TailExponentEstimator()
theory = TheoryPredictors()

sizes = np.array([mo.total_size for mo in metaorders])
beta = tail_estimator.hill_estimator(sizes)
ggps_pred = theory.ggps_delta(beta)

print(f"Measured delta:        {fit_result['delta']:.3f}")
print(f"beta (Hill, size tail): {beta:.3f} -> GGPS predicts delta={ggps_pred:.3f}")
print(f"Paper's finding: GGPS over-predicts by roughly a factor of ~2 vs measured delta ~0.5")

## 4. Mini counterfactual ablation (baseline vs. no-splitting, demo scale)

Reproduces the *direction* of Table 1's headline finding on a much smaller scale: removing order splitting should collapse delta well below 0.5.

In [ ]:
def run_and_fit(scenario_name, seed=1):
    sc = apply_scenario(sampler.sample(n_stocks=1, seed=seed)[0], scenario_name)
    sim = StockMarketSimulation(sc.to_dict(), cfg['model'], cfg['simulation'])
    res = sim.run(n_steps=N_STEPS_DEMO, warmup_steps=WARMUP_DEMO)
    dv, dr = res.daily_volume_and_range()
    mos = reconstructor.reconstruct(res.trade_tape, delta_t_ticks=cfg['data']['metaorder_delta_t_ticks'], steps_per_day=res.steps_per_day)
    q, i = reconstructor.normalize(mos, dv, dr, cfg['data']['min_qnorm_threshold_pct'])
    try:
        fr = fitter.log_bin_and_fit(q, i, n_bins=cfg['data']['log_bins'], min_pts=5)
        return fr['delta']
    except ValueError:
        return float('nan')

baseline_delta = run_and_fit('baseline')
no_splitting_delta = run_and_fit('no_splitting')

print(f"baseline:     delta={baseline_delta:.3f}  (paper: 0.549)")
print(f"no_splitting: delta={no_splitting_delta:.3f}  (paper: 0.324)")
print("\nAt demo scale (60k steps vs paper's 1M), expect much noisier estimates than the paper's\n"
      "20-stock averages -- run run_counterfactual_suite.py for the full-scale reproduction.")

## Next steps

- Full-scale baseline: `python train.py --config ../configs/config.yaml` (2000 stocks, ~hours depending on cores)
- Full Table 1 / Figure 7: `python run_counterfactual_suite.py --config ../configs/config.yaml`
- See `README.md`'s *Reproducibility Notes* for which unstated parameters most affect how closely your numbers match the paper's.